In [1]:
import os, math, json, random, warnings, gc, hashlib
warnings.filterwarnings('ignore')

import numpy as np
import cv2
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from tqdm import tqdm
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

try:
    import torchvision.models as tv_models
    import torchvision.transforms as T_vis
    HAS_TV = True
except ImportError:
    HAS_TV = False
    print("[WARN] torchvision not installed: pip install torchvision")

try:
    import seaborn as sns
    HAS_SNS = True
except ImportError:
    HAS_SNS = False

class Config:
    HIDDEN_DIM   = 128       # 128 hidden nodes of attn
    NUM_HEADS    = 4         # four attn heads
    FF_DIM       = 128       # feed-forward layer of dim 128

    # Emotion classes
    # Unified MOSEI space: happy=0  sad=1  anger=2  surprise=3  disgust=4  fear=5
    EMOTIONS     = ['happy', 'sad', 'anger', 'surprise', 'disgust', 'fear']
    NUM_EMOTIONS = 6

    # Feature dim
    FEAT_DIM     = 300       # fully-connected layer with 300 hidden units

    # Loss weights 
    VAE_W        = 1.0
    GAN_W        = 1.0
    SEQDIS_W     = 1.0
    CYCLE_W      = 5.0     

    # Training
    BATCH_SIZE       = 64      
    LR_CLASSIFIER    = 3e-4  
    LR_TRANSLATOR    = 1e-4    
    LR               = 3e-4    
    EPOCHS           = 10
    GRAD_CLIP        = 1.0
    E2E_RATIO        = 7       
    # Warmup: classifier-only for first N epochs so translators stabilise
    # before their noisy KL/GAN/cycle losses contaminate classifier gradients.
    WARMUP_EPOCHS    = 3
    MAX_SEQ          = 30     
    SAMPLE_RATE      = 16_000
    LATENT_DIM       = 128

    MOSI_PATH    = '/kaggle/input/datasets/mathurinache/cmu-mosi'
    CKP_PATH     = '/kaggle/input/datasets/shuvoalok/ck-dataset'
    RAVDESS_PATH = '/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-song-audio'
    SEMEVAL_PATH = '/kaggle/input/datasets/context/semeval-2018-task-ec'

    CACHE_DIR    = '/kaggle/working/feat_cache'   
    GLOVE_CACHE  = '/kaggle/working/glove_cache'

    VIS_BACKEND  = 'vggface2'
    GLOVE_NAME   = '6B'

    DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


cfg = Config()
os.makedirs(cfg.CACHE_DIR,   exist_ok=True)
os.makedirs(cfg.GLOVE_CACHE, exist_ok=True)

print(f"Device       : {cfg.DEVICE}")
print(f"Batch / LR   : {cfg.BATCH_SIZE} / {cfg.LR}")
print(f"E2E ratio    : 1:{cfg.E2E_RATIO}")
print(f"Visual back  : {cfg.VIS_BACKEND}")
print(f"GloVe        : {cfg.GLOVE_NAME} 300-d")

CKP_LABEL = {
    'happy'   : 0,
    'sadness' : 1,
    'anger'   : 2,
    'surprise': 3,
    'disgust' : 4,
    'fear'    : 5,
    'contempt': -1,    # no MOSEI equivalent
}

RAVDESS_LABEL = {
    1: -1,   # neutral (skip)
    2: -1,   # calm    (skip)
    3:  0,   # happy
    4:  1,   # sad
    5:  2,   # angry
    6:  5,   # fearful
    7:  4,   # disgust
    8:  3,   # surprised
}

SEMEVAL_LABEL = {
    'joy'     : 0,
    'sadness' : 1,
    'anger'   : 2,
    'surprise': 3,
    'disgust' : 4,
    'fear'    : 5,
}

# CMU-MOSI  —  continuous sentiment [-3,+3]  (emotion index)
def mosi_score_to_label(score: float) -> int:
    """
    Strong positive sentiment  → happy (0)
    Strong negative sentiment  → sad   (1)
    Weak / ambiguous sentiment → skip  (-1)
    """
    if   score >  1.0: return 0
    elif score < -1.0: return 1
    else:              return -1


_IMG_EXTS   = {'.png', '.jpg', '.jpeg', '.bmp'}
_AUDIO_EXTS = {'.wav', '.mp3', '.flac'}

def _cache_path(src, tag):
    h = hashlib.md5(str(src).encode()).hexdigest()[:14]
    return os.path.join(cfg.CACHE_DIR, f'{tag}_{h}.npy')

def _zeros(T=None):
    return np.zeros((T or cfg.MAX_SEQ, cfg.FEAT_DIM), np.float32)

class VisualExtractor:
    def __init__(self, device):
        self.device   = device
        self.mode     = None
        self.img_size = 224
        self._init_backbone()
        self._init_face_detector()

    # backbone selection
    def _init_backbone(self):
        prefer = cfg.VIS_BACKEND

        # Option 1: VGGFace2 InceptionResnetV1
        if prefer == 'vggface2':
            try:
                from facenet_pytorch import InceptionResnetV1, MTCNN
                self.mtcnn     = MTCNN(image_size=160, device=self.device,
                                       keep_all=False, post_process=False,
                                       min_face_size=20)
                base           = InceptionResnetV1(pretrained='vggface2').eval()
                self.backbone  = nn.Sequential(
                    *list(base.children())[:-1]).to(self.device).eval()
                self.proj      = nn.Linear(512, cfg.FEAT_DIM).to(self.device)
                nn.init.xavier_uniform_(self.proj.weight)
                self.img_size  = 160
                self.mode      = 'vggface2'
                print("Visual : VGGFace2 InceptionResnetV1 + MTCNN")
                return
            except ImportError:
                print("  facenet-pytorch not found (trying ResNet-50)")
                print("  (For best results: pip install facenet-pytorch)")

        # Option 2: ResNet-50 ImageNet
        if HAS_TV and prefer in ('vggface2', 'resnet50'):
            try:
                bb     = tv_models.resnet50(
                    weights=tv_models.ResNet50_Weights.IMAGENET1K_V1)
                bb.fc  = nn.Linear(2048, cfg.FEAT_DIM)
                nn.init.xavier_uniform_(bb.fc.weight)
                for n, p in bb.named_parameters():
                    p.requires_grad = 'fc' in n
                self.backbone = bb.to(self.device).eval()
                self.norm     = T_vis.Normalize([0.485,0.456,0.406],
                                               [0.229,0.224,0.225])
                self.mode     = 'resnet50'
                print("Visual : ResNet-50 ImageNet + 300-d head")
                return
            except Exception as e:
                print(f"  ResNet-50 failed ({e}) → trying ResNet-18")

        # Option 3: ResNet-18 ImageNet
        if HAS_TV:
            bb    = tv_models.resnet18(
                weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
            bb.fc = nn.Linear(512, cfg.FEAT_DIM)
            nn.init.xavier_uniform_(bb.fc.weight)
            for n, p in bb.named_parameters():
                p.requires_grad = 'fc' in n
            self.backbone = bb.to(self.device).eval()
            self.norm     = T_vis.Normalize([0.485,0.456,0.406],
                                           [0.229,0.224,0.225])
            self.mode     = 'resnet18'
            print("Visual : ResNet-18 ImageNet + 300-d head")
            return

        self.backbone = nn.Sequential(
            nn.Conv2d(3,32,3,2,1), nn.ReLU(),
            nn.Conv2d(32,64,3,2,1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1)), nn.Flatten(),
            nn.Linear(64, cfg.FEAT_DIM)
        ).to(self.device).eval()
        self.norm     = None
        self.mode     = 'scratch'
        print(" Visual : scratch CNN  (pip install torchvision for better)")

    def _init_face_detector(self):
        haar = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        self.haar = cv2.CascadeClassifier(haar)

    # image preprocessing 
    def _crop_face(self, bgr, size):
        gray  = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        faces = self.haar.detectMultiScale(gray, 1.1, 5, minSize=(20,20))
        if len(faces):
            x, y, w, h = max(faces, key=lambda r: r[2]*r[3])
            bgr = bgr[y:y+h, x:x+w]
        return cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), (size, size))

    # single-frame encoder 
    @torch.no_grad()
    def _encode_frame(self, bgr):
        if self.mode == 'vggface2':
            try:
                from PIL import Image as PILImage
                pil  = PILImage.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
                face = self.mtcnn(pil)                # (3,160,160) or None
                if face is None:
                    # MTCNN found no face (use the whole frame)
                    face = T_vis.functional.to_tensor(
                        pil.resize((160,160))).to(self.device)
                else:
                    face = face.to(self.device)
                emb = self.backbone(face.unsqueeze(0))  # (1,512,1,1) or (1,512)
                if emb.dim() == 4:
                    emb = emb.squeeze(-1).squeeze(-1)
                return self.proj(emb).squeeze(0).cpu().numpy()
            except Exception:
                return np.zeros(cfg.FEAT_DIM, np.float32)

        # ResNet / scratch path
        rgb = self._crop_face(bgr, self.img_size)
        t   = torch.FloatTensor(rgb / 255.).permute(2, 0, 1)
        if hasattr(self, 'norm') and self.norm is not None:
            t = self.norm(t)
        return self.backbone(
            t.unsqueeze(0).to(self.device)).squeeze(0).cpu().numpy()

    @torch.no_grad()
    def extract_video(self, video_path, T=None):
        T  = T or cfg.MAX_SEQ
        ck = _cache_path(video_path, f'vis_{self.mode}')
        if os.path.exists(ck): return np.load(ck)

        cap   = cv2.VideoCapture(video_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total == 0:
            cap.release()
            out = _zeros(T); np.save(ck, out); return out

        feats = []
        for idx in np.linspace(0, total-1, T, dtype=int):
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, bgr = cap.read()
            feats.append(self._encode_frame(bgr) if ret
                         else np.zeros(cfg.FEAT_DIM, np.float32))
        cap.release()
        out = np.stack(feats)
        np.save(ck, out); return out

    @torch.no_grad()
    def extract_image(self, image_path, T=None):
        T  = T or cfg.MAX_SEQ
        ck = _cache_path(image_path, f'vis_{self.mode}_img')
        if os.path.exists(ck): return np.load(ck)

        bgr = cv2.imread(image_path)
        if bgr is None:
            out = _zeros(T); np.save(ck, out); return out

        vec = self._encode_frame(bgr)            # (FEAT_DIM,)
        out = np.tile(vec, (T, 1))               # (T, FEAT_DIM)
        np.save(ck, out); return out


class _ConvBlock(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ci, co, 3, 1, 1, bias=False),
            nn.BatchNorm2d(co), nn.ReLU(),
            nn.Conv2d(co, co, 3, 1, 1, bias=False),
            nn.BatchNorm2d(co), nn.ReLU())
    def forward(self, x): return self.net(x)


class CNN14(nn.Module):
    def __init__(self, feat_dim=300):
        super().__init__()
        self.bn0  = nn.BatchNorm2d(1)
        self.b1   = _ConvBlock(1,     64)
        self.b2   = _ConvBlock(64,   128)
        self.b3   = _ConvBlock(128,  256)
        self.b4   = _ConvBlock(256,  512)
        self.b5   = _ConvBlock(512, 1024)
        self.b6   = _ConvBlock(1024,2048)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1  = nn.Linear(2048, 2048)
        self.drop = nn.Dropout(0.2)
        self.proj = nn.Linear(2048, feat_dim)   # 300-d output

    @staticmethod
    def _safe_pool(x):
        kh = 2 if x.shape[2] >= 2 else 1
        kw = 2 if x.shape[3] >= 2 else 1
        if kh == 1 and kw == 1:
            return x
        return F.avg_pool2d(x, (kh, kw))

    def forward(self, x):               # x: (B, 1, n_mels, time)
        x = self.bn0(x)
        x = self._safe_pool(self.b1(x))
        x = self._safe_pool(self.b2(x))
        x = self._safe_pool(self.b3(x))
        x = self._safe_pool(self.b4(x))
        x = self._safe_pool(self.b5(x))
        x = self._safe_pool(self.b6(x))
        x = self.pool(x).squeeze(-1).squeeze(-1)   # AdaptiveAvgPool2d(1,1)
        x = F.relu(self.fc1(self.drop(x)))
        return self.proj(x)             # (B, feat_dim)


_PANNS_URL  = ("https://zenodo.org/record/3987831/files/"
               "Cnn14_mAP%3D0.431.pth?download=1")
_PANNS_FILE = "Cnn14_mAP=0.431.pth"

# PANNs checkpoint
_PANNS_KEY_MAP = {
    'conv_block1':'b1','conv_block2':'b2','conv_block3':'b3',
    'conv_block4':'b4','conv_block5':'b5','conv_block6':'b6'}


def _download_panns(cache_dir: str) -> str | None:
    path = os.path.join(cache_dir, _PANNS_FILE)
    if os.path.exists(path):
        print(f" PANNs weights already cached: {path}")
        return path
    try:
        import urllib.request
        print("  Downloading PANNs CNN14 from Zenodo…")
        urllib.request.urlretrieve(_PANNS_URL, path)
        print(f" Saved → {path}"); return path
    except Exception as e:
        print(f" Zenodo download failed: {e}"); return None


def _load_panns(model: CNN14, ckpt_path: str) -> int:
    ckpt  = torch.load(ckpt_path, map_location='cpu')
    state = ckpt.get('model', ckpt)

    remapped = {}
    for k, v in state.items():
        nk = k
        for old, new in _PANNS_KEY_MAP.items():
            nk = nk.replace(old, new)
        nk = (nk.replace('.conv1.', '.net.0.')
                .replace('.conv2.', '.net.3.')
                .replace('.bn1.',   '.net.1.')
                .replace('.bn2.',   '.net.4.'))
        remapped[nk] = v

    own   = model.state_dict()
    match = {k: v for k, v in remapped.items()
             if k in own and own[k].shape == v.shape}
    own.update(match)
    model.load_state_dict(own, strict=False)
    pct = 100 * len(match) / max(len(own), 1)
    print(f" PANNs weights: {len(match)}/{len(own)} layers loaded ({pct:.0f}%)")
    return len(match)


class AudioExtractor:
    def __init__(self, device):
        self.device = device
        self.n_mels = 128
        self.hop    = 512     

        self.cnn14  = CNN14(feat_dim=cfg.FEAT_DIM).to(device)
        path        = _download_panns(cfg.CACHE_DIR)
        if path:
            n = _load_panns(self.cnn14, path)
            tag = "ACTUAL weights" if n > 50 else "partial match"
            print(f"Audio  : PANNs CNN14 ({tag}) + log-Mel 128 bins")
        else:
            print(" Audio : CNN14 random init (enable Kaggle internet to "
                  "download Zenodo weights)")
        self.cnn14.eval()

    @torch.no_grad()
    def extract(self, audio_path: str, T: int = None) -> np.ndarray:
        T  = T or cfg.MAX_SEQ
        ck = _cache_path(audio_path, 'aud')
        if os.path.exists(ck): return np.load(ck)

        try:
            y, _ = librosa.load(audio_path, sr=cfg.SAMPLE_RATE, mono=True)
        except Exception as e:
            out = _zeros(T); np.save(ck, out); return out

        # log-Mel spectrogram — LEAF approximation
        mel = librosa.power_to_db(
            librosa.feature.melspectrogram(
                y=y, sr=cfg.SAMPLE_RATE, n_mels=self.n_mels,
                hop_length=self.hop, fmin=50, fmax=8000),
            ref=np.max)                         # (128, n_frames)

        n_frames = mel.shape[1]
        if n_frames == 0:
            out = _zeros(T); np.save(ck, out); return out

        # Split into T equal chunks (T frame embeddings)
        bounds = np.linspace(0, n_frames, T + 1, dtype=int)
        feats  = []
        # CNN14 needs time-dim >= 64 .
        # Short clips or many chunks can produce tiny slices — pad to 64.
        MIN_FRAMES = 64
        for i in range(T):
            ch = mel[:, bounds[i]:bounds[i+1]]
            if ch.shape[1] == 0:
                feats.append(np.zeros(cfg.FEAT_DIM, np.float32)); continue
            if ch.shape[1] < MIN_FRAMES:
                ch = np.pad(ch, ((0,0),(0, MIN_FRAMES - ch.shape[1])))
            t = (torch.FloatTensor(ch)
                 .unsqueeze(0).unsqueeze(0)     # (1,1,128,chunk)
                 .to(self.device))
            feats.append(self.cnn14(t).squeeze(0).cpu().numpy())

        out = np.stack(feats)
        np.save(ck, out); return out

_GLOVE_URLS = {
    '6B':   'https://nlp.stanford.edu/data/glove.6B.zip',
    '840B': 'https://nlp.stanford.edu/data/glove.840B.300d.zip',
}
_GLOVE_TXT = {
    '6B':   'glove.6B.300d.txt',
    '840B': 'glove.840B.300d.txt',
}


def _download_glove(name: str, cache_dir: str) -> str | None:
    
    import urllib.request, zipfile
    txt_name = _GLOVE_TXT[name]
    txt_path = os.path.join(cache_dir, txt_name)
    if os.path.exists(txt_path):
        print(f"  GloVe {name} 300-d already cached: {txt_path}")
        return txt_path

    zip_path = os.path.join(cache_dir, f'glove.{name}.zip')
    url      = _GLOVE_URLS[name]
    size_str = '~820 MB' if name == '6B' else '~2 GB'

    # Download zip
    if not os.path.exists(zip_path):
        print(f"  Downloading GloVe {name} 300-d from Stanford NLP "
              f"({size_str}) …")
        print(f"  URL: {url}")
        try:
            def _progress(count, block, total):
                if total > 0 and count % 2000 == 0:
                    pct = min(100, count * block * 100 // total)
                    print(f"\r  Progress: {pct}%", end='', flush=True)
            urllib.request.urlretrieve(url, zip_path, _progress)
            print()   # newline after progress
            print(f" Downloaded → {zip_path}")
        except Exception as e:
            print(f" GloVe download failed: {e}")
            return None

    print(f"  Extracting {txt_name} from zip …")
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            members = [m for m in zf.namelist() if '300d' in m]
            target  = members[0] if members else txt_name
            zf.extract(target, cache_dir)
            extracted = os.path.join(cache_dir, target)
            if extracted != txt_path and os.path.exists(extracted):
                os.rename(extracted, txt_path)
        print(f" Extracted → {txt_path}")
        # Remove zip
        try: os.remove(zip_path)
        except Exception: pass
        return txt_path
    except Exception as e:
        print(f" Extraction failed: {e}"); return None


def _build_glove_index(txt_path: str, cache_dir: str,
                       name: str) -> tuple[dict, np.ndarray] | tuple[None, None]:

    w2i_path = os.path.join(cache_dir, f'glove_{name}_w2i.json')
    vec_path = os.path.join(cache_dir, f'glove_{name}_vecs.npy')

    if os.path.exists(w2i_path) and os.path.exists(vec_path):
        print(f"  Loading cached GloVe index …")
        word2idx = json.load(open(w2i_path))
        vectors  = np.load(vec_path)
        print(f" Text   : GloVe {name} 300-d  ({len(word2idx):,} words, "
              f"loaded from cache)")
        return word2idx, vectors

    print(f"  Parsing GloVe vectors…")
    words, vecs = [], []
    try:
        with open(txt_path, encoding='utf-8', errors='ignore') as f:
            for i, line in enumerate(f):
                parts = line.rstrip().split(' ')
                if len(parts) != 301: continue
                words.append(parts[0])
                vecs.append(np.array(parts[1:], dtype=np.float32))
                if i % 100_000 == 0 and i > 0:
                    print(f"\r  Parsed {i:,} lines …", end='', flush=True)
        print()
    except Exception as e:
        print(f" Parse error: {e}"); return None, None

    vectors  = np.stack(vecs)
    word2idx = {w: i for i, w in enumerate(words)}

    print(f" Saving index cache …")
    np.save(vec_path, vectors)
    json.dump(word2idx, open(w2i_path, 'w'))
    print(f" Text   : GloVe {name} 300-d  ({len(word2idx):,} words)")
    return word2idx, vectors


class TextExtractor:
    def __init__(self):
        self.word2idx = None
        self.vectors  = None
        self._mem     = {}   

        name     = cfg.GLOVE_NAME
        txt_path = _download_glove(name, cfg.GLOVE_CACHE)
        if txt_path:
            self.word2idx, self.vectors = _build_glove_index(
                txt_path, cfg.GLOVE_CACHE, name)

        if self.word2idx is None:
            print("Text   : random OOV vectors (GloVe unavailable). "
                  "Enable Kaggle internet and re-run.")

    def _word_vec(self, word: str) -> np.ndarray:
        w = word.lower().strip()
        if not w: return np.zeros(300, np.float32)
        if w in self._mem: return self._mem[w]

        if self.word2idx is not None:
            idx = self.word2idx.get(w)
            if idx is not None:
                v = self.vectors[idx]              # already float32 np array
                self._mem[w] = v; return v

        # Out-of-vocabulary: seeded deterministic random
        np.random.seed(abs(hash(w)) % 2**31)
        v = (np.random.randn(300) * 0.1).astype(np.float32)
        self._mem[w] = v; return v

    def extract(self, text: str, T: int = None,
                cache_key: str = None) -> np.ndarray:
        T = T or cfg.MAX_SEQ
        if cache_key:
            ck = _cache_path(cache_key, 'txt')
            if os.path.exists(ck): return np.load(ck)

        words = str(text).lower().split()[:T]
        feats = [self._word_vec(w) for w in words]
        while len(feats) < T:
            feats.append(np.zeros(300, np.float32))
        out = np.stack(feats)[:, :cfg.FEAT_DIM]

        if cache_key:
            np.save(_cache_path(cache_key, 'txt'), out)
        return out


# 3D  EXTRACTOR SHAPE + SANITY VERIFICATION
def verify_extractors(vis: VisualExtractor,
                      aud: AudioExtractor,
                      txt: TextExtractor) -> bool:
    import tempfile
    import scipy.io.wavfile as wav_io

    T, D, ok = cfg.MAX_SEQ, cfg.FEAT_DIM, True

    # Visual — dummy video
    tmp_v = tempfile.mktemp(suffix='.mp4')
    wr = cv2.VideoWriter(tmp_v, cv2.VideoWriter_fourcc(*'mp4v'),
                         25, (160, 160))
    for _ in range(35): wr.write(np.zeros((160, 160, 3), np.uint8))
    wr.release()
    vf = vis.extract_video(tmp_v, T)
    os.unlink(tmp_v)
    ok &= vf.shape == (T, D)
    print(f"  Visual (video) : {vf.shape}  "
          f"{'' if vf.shape == (T,D) else 'SHAPE MISMATCH'}")

    # Visual — dummy image (CK+ path)
    tmp_img = tempfile.mktemp(suffix='.png')
    cv2.imwrite(tmp_img, np.zeros((80, 80, 3), np.uint8))
    vf2 = vis.extract_image(tmp_img, T)
    os.unlink(tmp_img)
    ok &= vf2.shape == (T, D)
    print(f"  Visual (image) : {vf2.shape}  "
          f"{'' if vf2.shape == (T,D) else 'SHAPE MISMATCH'}")

    # Audio — silence wav
    tmp_a = tempfile.mktemp(suffix='.wav')
    wav_io.write(tmp_a, cfg.SAMPLE_RATE,
                 np.zeros(cfg.SAMPLE_RATE * 3, np.int16))
    af = aud.extract(tmp_a, T)
    os.unlink(tmp_a)
    ok &= af.shape == (T, D)
    print(f"  Audio          : {af.shape}  "
          f"{'' if af.shape == (T,D) else 'SHAPE MISMATCH'}")

    # Text
    tf = txt.extract("I am feeling very happy today", T)
    ok &= tf.shape == (T, D)
    print(f"  Text           : {tf.shape}  "
          f"{'' if tf.shape == (T,D) else 'SHAPE MISMATCH'}")

    if txt.word2idx is not None:
        h  = txt._word_vec('happy')
        s  = txt._word_vec('sad')
        sim = float(np.dot(h, s) /
                    (np.linalg.norm(h) * np.linalg.norm(s) + 1e-8))
        flag = '' if sim > 0.30 else 'real GloVe gives ~0.56'
        print(f"  GloVe sanity   : cos_sim(happy, sad) = {sim:.3f}  {flag}")

    result = "All checks passed" if ok else "ISSUES FOUND — fix before training"
    print(f"  {result}")
    return ok


def _mk(vid_id, video_path, audio_path, text_path, label,
        hv, ha, ht, aligned):
    T, D = cfg.MAX_SEQ, cfg.FEAT_DIM
    return dict(
        video_id   = vid_id,
        video_path = video_path,
        audio_path = audio_path,
        text_path  = text_path,
        label      = label,
        has_visual = hv,
        has_audio  = ha,
        has_text   = ht,
        has_aligned= aligned,
        visual     = torch.zeros(T, D),
        audio      = torch.zeros(T, D),
        text       = torch.zeros(T, D))


# 4A  CMU-MOSI  (trimodal) 
class CMUMOSIDataset(Dataset):

    def __init__(self, base_path: str, split: str = 'train'):
        self.base    = base_path
        self.split   = split
        self.samples = []
        self._collect()
        self._split_samples()
        print(f"CMU-MOSI  [{split:5s}]  {len(self):5d} samples  "
              f"(trimodal: video+audio+text)")

    def _collect(self):
        video_root = os.path.join(self.base, 'Raw', 'Video')
        audio_root = os.path.join(self.base, 'Raw', 'Audio')
        text_root  = os.path.join(self.base, 'Raw', 'Transcript')

        if not os.path.isdir(video_root):
            print(f" CMU-MOSI: no Video dir found at {video_root}")
            return

        # Load labels.json
        label_map = {}
        for lf in [os.path.join(self.base, 'labels.json'),
                   os.path.join(self.base, 'Raw', 'labels.json')]:
            if os.path.exists(lf):
                label_map = json.load(open(lf)); break

        # Walk video root — handles both flat and nested layouts
        mp4_files = []
        for dirpath, _, files in os.walk(video_root):
            for f in files:
                if f.endswith('.mp4'):
                    mp4_files.append(os.path.join(dirpath, f))

        for vp in mp4_files:
            vid_id = os.path.splitext(os.path.basename(vp))[0]

            # Resolve label
            lbl = -1
            if vid_id in label_map:
                raw = label_map[vid_id]
                if isinstance(raw, (int, float)):
                    lbl = mosi_score_to_label(float(raw))
                elif isinstance(raw, str):
                    lbl = {'happy':0,'sad':1,'anger':2,
                           'surprise':3,'disgust':4,'fear':5}.get(
                        raw.lower(), -1)
                elif isinstance(raw, list) and raw:
                    # some label files store list of scores per segment
                    lbl = mosi_score_to_label(float(np.mean(raw)))
            else:
                # No label → placeholder 
                lbl = abs(hash(vid_id)) % cfg.NUM_EMOTIONS

            if lbl < 0:
                continue   # ambiguous 

            ap = self._search(audio_root, vid_id,
                              _AUDIO_EXTS | {'.mp4'})
            tp = self._search(text_root,  vid_id,
                              {'.txt', '.textonly', '.annotprocessed', ''})

            self.samples.append(_mk(
                vid_id, vp, ap, tp, lbl,
                True,                          # has_visual  always True
                ap is not None,                # has_audio
                tp is not None,                # has_text
                ap is not None and tp is not None))  # has_aligned

    @staticmethod
    def _search(root: str, vid_id: str, exts: set) -> str | None:
        if not os.path.isdir(root): return None
        for dirpath, _, files in os.walk(root):
            for f in files:
                name, ext = os.path.splitext(f)
                if name == vid_id and (ext.lower() in exts or ext == ''):
                    return os.path.join(dirpath, f)
        return None

    def _split_samples(self):
        random.shuffle(self.samples)
        n = len(self.samples)
        self.samples = {
            'train': self.samples[:int(n * 0.70)],
            'val':   self.samples[int(n * 0.70):int(n * 0.85)],
            'test':  self.samples[int(n * 0.85):]
        }[self.split]

    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return dict(self.samples[i])


# 4B  CK+  (unimodal: visual only)
class CKPlusDataset(Dataset):
    def __init__(self, base_path: str, split: str = 'train'):
        self.base    = base_path
        self.split   = split
        self.samples = []
        self._collect()
        self._split_samples()
        print(f"CK+       [{split:5s}]  {len(self):5d} samples  "
              f"(visual only, flat emotion folders)")

    def _collect(self):
        if not os.path.isdir(self.base):
            print(f" CK+: path not found: {self.base}"); return

        for folder_name in os.listdir(self.base):
            lbl = CKP_LABEL.get(folder_name.lower(), -1)
            if lbl < 0: continue              

            folder_path = os.path.join(self.base, folder_name)
            if not os.path.isdir(folder_path): continue

            for fname in os.listdir(folder_path):
                ext = os.path.splitext(fname)[1].lower()
                if ext not in _IMG_EXTS: continue

                img_path = os.path.join(folder_path, fname)
                vid_id   = f'ckp_{folder_name}_{os.path.splitext(fname)[0]}'
                self.samples.append(_mk(
                    vid_id,
                    img_path, 
                    None, None, lbl,
                    True, False, False, False))

    def _split_samples(self):
        random.shuffle(self.samples)
        n = len(self.samples)
        self.samples = {
            'train': self.samples[:int(n * 0.70)],
            'val':   self.samples[int(n * 0.70):int(n * 0.85)],
            'test':  self.samples[int(n * 0.85):]
        }[self.split]

    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return dict(self.samples[i])


# 4C  RAVDESS  (unimodal: audio only, song version)
class RAVDESSDataset(Dataset):
    def __init__(self, base_path: str, split: str = 'train'):
        self.base    = base_path
        self.split   = split
        self.samples = []
        self._collect()
        self._split_samples()
        print(f"RAVDESS   [{split:5s}]  {len(self):5d} samples  "
              f"(audio only, song modality)")

    def _collect(self):
        if not os.path.isdir(self.base):
            print(f" RAVDESS: path not found: {self.base}"); return

        for actor_dir in sorted(os.listdir(self.base)):
            if not actor_dir.startswith('Actor_'): continue
            ad = os.path.join(self.base, actor_dir)
            if not os.path.isdir(ad): continue

            for fname in os.listdir(ad):
                ext = os.path.splitext(fname)[1].lower()
                if ext not in _AUDIO_EXTS: continue

                # Parse RAVDESS filename: field[2] (0-indexed) = emotion
                fields = os.path.splitext(fname)[0].split('-')
                if len(fields) < 7: continue
                try:
                    emo_code = int(fields[2])
                except ValueError:
                    continue

                lbl = RAVDESS_LABEL.get(emo_code, -1)
                if lbl < 0: continue   

                ap = os.path.join(ad, fname)
                self.samples.append(_mk(
                    fname, None, ap, None, lbl,
                    False, True, False, False))

    def _split_samples(self):
        random.shuffle(self.samples)
        n = len(self.samples)
        self.samples = {
            'train': self.samples[:int(n * 0.70)],
            'val':   self.samples[int(n * 0.70):int(n * 0.85)],
            'test':  self.samples[int(n * 0.85):]
        }[self.split]

    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return dict(self.samples[i])


# 4D  SEMEVAL 2018 Task E-c  (unimodal: text only)
class SemEvalDataset(Dataset):
    
    _FILEMAP = {
        'train': '2018-E-c-En-train.txt',
        'val':   '2018-E-c-En-dev.txt',
        'test':  '2018-E-c-En-test.txt',
    }

    def __init__(self, base_path: str, split: str = 'train'):
        fname = self._FILEMAP.get(split, self._FILEMAP['train'])
        fpath = os.path.join(base_path, fname)
        self.samples = []
        self._collect(fpath)
        print(f"SemEval18 [{split:5s}]  {len(self):5d} samples  "
              f"(text only, {fname})")

    def _collect(self, fpath: str):
        if not os.path.exists(fpath):
            print(f" SemEval: file not found: {fpath}"); return

        with open(fpath, encoding='utf-8', errors='ignore') as f:
            header = f.readline().strip().split('\t')
            for line in f:
                cols = line.strip().split('\t')
                if len(cols) < 3: continue

                tweet = cols[1]
                lbl   = -1
                # Use first positive column that maps to MOSEI
                for col_name, mosei_idx in SEMEVAL_LABEL.items():
                    if col_name in header:
                        ci = header.index(col_name)
                        if ci < len(cols) and cols[ci].strip() == '1':
                            lbl = mosei_idx; break
                if lbl < 0: continue   # no MOSEI-compatible emotion

                self.samples.append(_mk(
                    cols[0],
                    None, None,
                    tweet, 
                    lbl,
                    False, False, True, False))

    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return dict(self.samples[i])


# 4E  Collate + Dataset Builder 
def collate_fn(batch: list) -> dict:
    return dict(
        visual      = torch.stack([b['visual']      for b in batch]),
        audio       = torch.stack([b['audio']       for b in batch]),
        text        = torch.stack([b['text']        for b in batch]),
        label       = torch.tensor([b['label']      for b in batch]),
        has_visual  = torch.tensor([b['has_visual'] for b in batch]),
        has_audio   = torch.tensor([b['has_audio']  for b in batch]),
        has_text    = torch.tensor([b['has_text']   for b in batch]),
        has_aligned = torch.tensor([b['has_aligned']for b in batch]),
        video_path  = [b['video_path']  for b in batch],
        audio_path  = [b['audio_path']  for b in batch],
        text_path   = [b['text_path']   for b in batch],
        video_id    = [b['video_id']    for b in batch])


def build_dataset(split: str) -> ConcatDataset | None:
    dsets, n_total = [], 0
    for cls, path, name in [
        (CMUMOSIDataset, cfg.MOSI_PATH,    'CMU-MOSI'),
        (CKPlusDataset,  cfg.CKP_PATH,     'CK+'),
        (RAVDESSDataset, cfg.RAVDESS_PATH, 'RAVDESS'),
        (SemEvalDataset, cfg.SEMEVAL_PATH, 'SemEval18'),
    ]:
        if not os.path.isdir(path):
            print(f" {name}: path not found → skipped"); continue
        try:
            ds = cls(path, split)
            if len(ds) > 0:
                dsets.append(ds); n_total += len(ds)
        except Exception as e:
            print(f" {name} error: {e}")

    if not dsets:
        return None
    print(f" Combined [{split:5s}]  total: {n_total} samples ")
    return ConcatDataset(dsets)


class PositionalEncoding(nn.Module):
    def __init__(self, d: int, dropout: float = 0.1, maxlen: int = 512):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe  = torch.zeros(maxlen, d)
        pos = torch.arange(maxlen).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d, 2).float() * (-math.log(10000.) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(x + self.pe[:, :x.size(1)])


# VAE Encoder  
class VAEEncoder(nn.Module):
    
    def __init__(self, in_dim, latent_dim,
                 shared_mu=None, shared_lv=None):
        super().__init__()
        self.private = nn.Sequential(
            nn.Linear(in_dim, 256), nn.LayerNorm(256), nn.LeakyReLU(0.2),
            nn.Linear(256, 128),    nn.LayerNorm(128), nn.LeakyReLU(0.2))
        self.mu = shared_mu or nn.Linear(128, latent_dim)
        self.lv = shared_lv or nn.Linear(128, latent_dim)

    def forward(self, x):
        h = self.private(x); return self.mu(h), self.lv(h)


# VAE Decoder  
class VAEDecoder(nn.Module):
    def __init__(self, latent_dim, out_dim, shared_out=None):
        super().__init__()
        self.private = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.LayerNorm(128), nn.LeakyReLU(0.2),
            nn.Linear(128, 256),        nn.LayerNorm(256), nn.LeakyReLU(0.2))
        self.out = shared_out or nn.Linear(256, out_dim)

    def forward(self, z): return self.out(self.private(z))


# Frame discriminator 
class FrameDisc(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 128), nn.LeakyReLU(0.2), nn.Dropout(0.1),
            nn.Linear(128, 64), nn.LeakyReLU(0.2),
            nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x)


# Sequence discriminator 
class SeqDisc(nn.Module):
    def __init__(self, feat_dim, K=6, d=128, h=4):
        super().__init__()
        self.proj = nn.Linear(feat_dim, d)
        self.pe   = PositionalEncoding(d)
        enc = nn.TransformerEncoderLayer(
            d, h, dim_feedforward=128, dropout=0.1, batch_first=True)
        self.tfm  = nn.TransformerEncoder(enc, num_layers=1)
        self.cls  = nn.Linear(d, K + 1)

    def forward(self, x):
        return self.cls(self.tfm(self.pe(self.proj(x))).mean(1))


# VAEGAN Translator
class VAEGANTranslator(nn.Module):
    def __init__(self, feat_dim, latent_dim, encoder, decoder,
                 K=6, name=''):
        super().__init__()
        self.D = feat_dim; self.name = name
        self.encoder = encoder; self.decoder = decoder
        self.fd = FrameDisc(feat_dim)
        self.sd = SeqDisc(feat_dim, K)

    @staticmethod
    def _reparam(mu, lv):
        return mu + torch.exp(0.5 * lv) * torch.randn_like(mu)

    def _enc(self, x):
        B, T, D = x.shape
        mu, lv  = self.encoder(x.reshape(B*T, D))
        return mu.reshape(B, T, -1), lv.reshape(B, T, -1)

    def _dec(self, z):
        B, T, L = z.shape
        return self.decoder(z.reshape(B*T, L)).reshape(B, T, -1)

    def translate(self, src):
        mu, lv = self._enc(src)
        return self._dec(self._reparam(mu, lv))

    def gen_losses(self, src, tgt, labels, aligned):
        mu, lv = self._enc(src)
        z      = self._reparam(mu, lv)
        fake   = self._dec(z)

        kl    = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
        recon = F.mse_loss(fake, tgt)

        # GAN loss
        if aligned.any():
            ff  = fake[aligned].reshape(-1, self.D)
            gan = -torch.mean(torch.log(self.fd(ff) + 1e-8))
        else:
            gan = src.new_zeros(1).squeeze()

        # SeqDis generator loss: fake features should predict real emotion class
        seq_g = F.cross_entropy(self.sd(fake), labels)

        return fake, dict(kl=kl, recon=recon, gan=gan, seq_g=seq_g)

    def disc_losses(self, src, tgt, labels, aligned):
        with torch.no_grad():
            fake = self._dec(self._reparam(*self._enc(src)))
        L = {}

        # Frame discriminator
        if aligned.any():
            rf  = tgt[aligned].reshape(-1, self.D).detach()
            ff  = fake[aligned].reshape(-1, self.D).detach()
            L['df'] = (-torch.mean(torch.log(self.fd(rf) + 1e-8))
                      - torch.mean(torch.log(1 - self.fd(ff) + 1e-8)))
        else:
            L['df'] = src.new_zeros(1).squeeze()

        # SeqDis
        fake_lbl = torch.full_like(labels, cfg.NUM_EMOTIONS)
        L['ds'] = (F.cross_entropy(self.sd(tgt.detach()),  labels)
                 + F.cross_entropy(self.sd(fake.detach()), fake_lbl))
        return L


# Modality Module 
class ModalityModule(nn.Module):
    def __init__(self, feat_dim, d=128, h=4):
        super().__init__()
        self.proj = nn.Linear(feat_dim, d)
        self.pe   = PositionalEncoding(d)
        enc = nn.TransformerEncoderLayer(
            d, h, dim_feedforward=128, dropout=0.1, batch_first=True)
        self.tfm  = nn.TransformerEncoder(enc, num_layers=2)
        self.norm = nn.LayerNorm(d)

    def forward(self, x):
        return self.norm(self.tfm(self.pe(self.proj(x))))   # (B,T,128)


# Multimodal Fusion Module
class FusionModule(nn.Module):

    def __init__(self, d=128, h=4, K=6):
        super().__init__()
        # register_buffer
        self.register_buffer('w', torch.tensor([1/3, 1/3, 1/3]))
        self.pe  = PositionalEncoding(d)
        enc = nn.TransformerEncoderLayer(
            d, h, dim_feedforward=128, dropout=0.1, batch_first=True)
        self.mfm = nn.TransformerEncoder(enc, num_layers=2)
        self.ff  = nn.Sequential(
            nn.Linear(d, d), nn.ReLU(), nn.Dropout(0.1), nn.Linear(d, K))

    def forward(self, a, v, t, hv=None, ha=None, ht=None):
        va = a.mean(1); vv = v.mean(1); vt = t.mean(1)   # each (B,128)
        if hv is not None and ha is not None and ht is not None:
            fa = ha.float().to(a.device).unsqueeze(1)     # (B,1)
            fv = hv.float().to(a.device).unsqueeze(1)
            ft = ht.float().to(a.device).unsqueeze(1)
            denom = (fa + fv + ft).clamp(min=1.0)
            ws = (fa*va + fv*vv + ft*vt) / denom
        else:
            ws = self.w[0]*va + self.w[1]*vv + self.w[2]*vt
        mf = self.mfm(self.pe(ws.unsqueeze(1))).squeeze(1)
        return self.ff(mf)                             # (B, K)


# cross-modal emotion model
class CrossModalEmotionModel(nn.Module):
    def __init__(self, c: Config):
        super().__init__()
        D, L = c.FEAT_DIM, c.LATENT_DIM

        # Shared last projection layers
        self.shared_mu   = nn.ModuleDict({m: nn.Linear(128, L) for m in 'avt'})
        self.shared_lv   = nn.ModuleDict({m: nn.Linear(128, L) for m in 'avt'})
        self.shared_dout = nn.ModuleDict({m: nn.Linear(256, D) for m in 'avt'})

        # 6 VAEGAN translators
        self.translators = nn.ModuleDict()
        for src in 'avt':
            for tgt in 'avt':
                if src == tgt: continue
                enc = VAEEncoder(D, L,
                                 self.shared_mu[src],
                                 self.shared_lv[src])
                dec = VAEDecoder(L, D, self.shared_dout[tgt])
                self.translators[f'{src}2{tgt}'] = VAEGANTranslator(
                    D, L, enc, dec, c.NUM_EMOTIONS, f'{src}2{tgt}')

        # Per-modality transformer encoders
        self.mm = nn.ModuleDict({m: ModalityModule(D) for m in 'avt'})

        # Fixed-weight multimodal fusion + classifier
        self.fusion = FusionModule(d=128, h=4, K=c.NUM_EMOTIONS)

        self.to(c.DEVICE)

    def classify(self, feats: dict,
                 hv=None, ha=None, ht=None) -> torch.Tensor:
        """Forward hv/ha/ht so FusionModule normalises by modalities present."""
        return self.fusion(
            self.mm['a'](feats['a']),
            self.mm['v'](feats['v']),
            self.mm['t'](feats['t']),
            hv=hv, ha=ha, ht=ht)

class Trainer:
    def __init__(self, model: CrossModalEmotionModel, c: Config,
                 vis: VisualExtractor,
                 aud: AudioExtractor,
                 txt: TextExtractor,
                 cls_w: torch.Tensor | None = None):
        self.model = model; self.c = c
        self.vis = vis; self.aud = aud; self.txt = txt

        # Separate param groups: lower LR for translators 
        trans_ids  = {id(p) for p in model.translators.parameters()}
        base_params  = [p for p in model.parameters() if id(p) not in trans_ids]
        trans_params = list(model.translators.parameters())
        self.opt = optim.Adam([
            {'params': base_params,  'lr': c.LR_CLASSIFIER},
            {'params': trans_params, 'lr': c.LR_TRANSLATOR},
        ])
        # CrossEntropyLoss — weighted by inverse class frequency
        # cls_w normalises loss contribution; combined with WeightedRandomSampler
        # this gives a strong, calibrated signal for minority classes.
        if cls_w is not None:
            self.crit = nn.CrossEntropyLoss(weight=cls_w.to(c.DEVICE))
        else:
            self.crit = nn.CrossEntropyLoss()
        self.sched = optim.lr_scheduler.ReduceLROnPlateau(
            self.opt, mode='max', factor=0.5, patience=3)

        self.best_val_wa = 0.
        self.global_step = 0
        self.hist = dict(tl=[], ta=[], va=[], vf=[])

    # batch feature extraction
    def _extract_batch(self, batch: dict) -> tuple:
        B        = len(batch['video_id'])
        T, D, dv = self.c.MAX_SEQ, self.c.FEAT_DIM, self.c.DEVICE
        V  = torch.zeros(B, T, D, device=dv)
        A  = torch.zeros_like(V)
        Tx = torch.zeros_like(V)

        for i in range(B):
            if batch['has_visual'][i]:
                vp = batch['video_path'][i]
                if vp:
                    try:
                        ext = os.path.splitext(vp)[1].lower()
                        if ext in _IMG_EXTS:
                            # CK+ single image
                            feat = self.vis.extract_image(vp, T)
                        else:
                            # CMU-MOSI video
                            feat = self.vis.extract_video(vp, T)
                        V[i] = torch.from_numpy(feat).to(dv)
                    except Exception:
                        pass   # leave zeros

            # Audio 
            if batch['has_audio'][i]:
                ap = batch['audio_path'][i]
                if ap and os.path.exists(ap):
                    try:
                        A[i] = torch.from_numpy(
                            self.aud.extract(ap, T)).to(dv)
                    except Exception:
                        pass

            # Text 
            if batch['has_text'][i]:
                tp = batch['text_path'][i]
                if tp:
                    try:
                        if os.path.isfile(str(tp)):
                            # CMU-MOSI transcript file
                            text     = open(tp, encoding='utf-8',
                                            errors='ignore').read()
                            ck_key   = tp
                        else:
                            # SemEval
                            text   = tp
                            ck_key = None
                        Tx[i] = torch.from_numpy(
                            self.txt.extract(text, T, ck_key)).to(dv)
                    except Exception:
                        pass
        return V, A, Tx

    # missing-modality augmentation
    @torch.no_grad()
    def _augment(self, feats: dict, hv, ha, ht,
                 use_translators: bool = True) -> dict:
        dev  = feats['v'].device
        out  = {k: v.clone() for k, v in feats.items()}
        if not use_translators:
            return out
        pres = {'v': hv.to(dev), 'a': ha.to(dev), 't': ht.to(dev)}

        for missing in 'avt':
            miss_mask = ~pres[missing]
            if not miss_mask.any(): continue
            for src in 'avt':
                if src == missing: continue
                can_fill = miss_mask & pres[src]
                if can_fill.any():
                    out[missing][can_fill] = (
                        self.model.translators[f'{src}2{missing}']
                        .translate(feats[src])[can_fill])
                    break  
        return out

    def _algorithm1(self, batch: dict) -> torch.Tensor:
        
        V, A, Tx  = self._extract_batch(batch)
        labels    = batch['label'].to(self.c.DEVICE)
        aligned   = batch['has_aligned'].to(self.c.DEVICE)
        present   = {'v': batch['has_visual'].to(self.c.DEVICE),
                     'a': batch['has_audio'].to(self.c.DEVICE),
                     't': batch['has_text'].to(self.c.DEVICE)}
        real      = {'v': V, 'a': A, 't': Tx}
        # Start with one real pool per modality
        pools     = {'v': [V], 'a': [A], 't': [Tx]}

        t_loss = real['a'].new_zeros(1).squeeze()

        for src in 'avt':
            if not present[src].any(): continue
            for tgt in 'avt':
                if tgt == src: continue

                tr      = self.model.translators[f'{src}2{tgt}']
                # GAN loss only where src AND tgt both have GT + aligned
                pair_al = aligned & present[src] & present[tgt]

                fake, gl = tr.gen_losses(real[src], real[tgt],
                                         labels, pair_al)
                dl       = tr.disc_losses(real[src], real[tgt],
                                          labels, pair_al)

                # Cycle loss
                back  = (self.model.translators[f'{tgt}2{src}']
                         .translate(fake))
                cycle = F.mse_loss(back, real[src]) * self.c.CYCLE_W

                t_loss = t_loss + (
                    self.c.VAE_W    * gl['kl']    +
                    self.c.VAE_W    * gl['recon'] +
                    self.c.GAN_W    * gl['gan']   +
                    self.c.SEQDIS_W * gl['seq_g'] +
                    dl['df'] + dl['ds'] + cycle)

                pools[tgt].append(fake.detach())

        # Combinatorial classifier loss (Algorithm 1 lines 28-32)
        # Every pool slot is filled (real or translated), so all-True
        # presence flags are semantically correct — no missing modalities.
        all_present = labels.new_ones(labels.shape[0], dtype=torch.bool)
        c_loss = real['a'].new_zeros(1).squeeze(); n = 0
        for af in pools['a']:
            for vf in pools['v']:
                for tf in pools['t']:
                    c_loss = c_loss + self.crit(
                        self.model.classify(
                            {'a': af, 'v': vf, 't': tf},
                            hv=all_present, ha=all_present, ht=all_present),
                        labels)
                    n += 1

        return t_loss + c_loss / max(n, 1)

    # train one epoch
    def train_epoch(self, loader: DataLoader, epoch: int) -> tuple:
        self.model.train()
        total, correct, n = 0., 0, 0
        in_warmup = (epoch <= self.c.WARMUP_EPOCHS)
        phase = (f'Warmup {epoch}/{self.c.WARMUP_EPOCHS}'
                 if in_warmup else f'E2E ep{epoch}')
        pbar = tqdm(loader, desc=f'Epoch {epoch:02d} [{phase}]', leave=True)

        for batch in pbar:
            self.global_step += 1
            hv = batch['has_visual']
            ha = batch['has_audio']
            ht = batch['has_text']

            if not in_warmup and self.global_step % self.c.E2E_RATIO == 0:
                loss = self._algorithm1(batch)
            else:
                V, A, Tx  = self._extract_batch(batch)
                labels    = batch['label'].to(self.c.DEVICE)
                # During warmup keep zeros for missing modalities:
                # random translator outputs are noisier than zeros.
                feats = self._augment(
                    {'v': V, 'a': A, 't': Tx}, hv, ha, ht,
                    use_translators=not in_warmup)

                logits = self.model.classify(
                    feats,
                    hv=hv.to(self.c.DEVICE),
                    ha=ha.to(self.c.DEVICE),
                    ht=ht.to(self.c.DEVICE))
                loss      = self.crit(logits, labels)
                _, pred   = logits.max(1)
                correct  += pred.eq(labels).sum().item()
                n        += labels.size(0)

            self.opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(),
                                     self.c.GRAD_CLIP)
            self.opt.step()
            total += loss.item()
            pbar.set_postfix(
                loss = f'{total / (pbar.n + 1):.4f}',
                acc  = f'{100 * correct / max(1, n):.1f}%')

        return total / len(loader), 100. * correct / max(1, n)

    # evaluate 
    @torch.no_grad()
    def evaluate(self, loader: DataLoader, tag: str = '') -> tuple:
        self.model.eval()
        preds, labs = [], []
        for batch in tqdm(loader, desc=f'Eval [{tag}]', leave=False):
            V, A, Tx = self._extract_batch(batch)
            labels   = batch['label'].to(self.c.DEVICE)
            hv = batch['has_visual']; ha = batch['has_audio']
            ht = batch['has_text']
            feats    = self._augment(
                {'v': V, 'a': A, 't': Tx}, hv, ha, ht,
                use_translators=True)
            logits   = self.model.classify(
                feats,
                hv=hv.to(self.c.DEVICE),
                ha=ha.to(self.c.DEVICE),
                ht=ht.to(self.c.DEVICE))
            preds   += logits.argmax(1).cpu().tolist()
            labs    += labels.cpu().tolist()

        # WA = Weighted Accuracy = standard accuracy
        wa = 100. * sum(p == l for p, l in zip(preds, labs)) / max(1, len(labs))
        f1 = f1_score(labs, preds, average='weighted', zero_division=0)
        print(f"  [{tag}]   WA = {wa:.2f}%    F1 = {f1:.4f}")
        print(classification_report(
            labs, preds, target_names=cfg.EMOTIONS, zero_division=0))
        return wa, f1, preds, labs

    # plots
    def save_confusion_matrix(self, preds, labs, title, fname):
        cm = confusion_matrix(labs, preds, normalize='true')
        fig, ax = plt.subplots(figsize=(8, 7))
        if HAS_SNS:
            sns.heatmap(cm, annot=True, fmt='.2f',
                        xticklabels=cfg.EMOTIONS,
                        yticklabels=cfg.EMOTIONS,
                        cmap='Blues', ax=ax)
        else:
            im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
            plt.colorbar(im, ax=ax)
            ax.set_xticks(range(len(cfg.EMOTIONS)))
            ax.set_xticklabels(cfg.EMOTIONS, rotation=45, ha='right')
            ax.set_yticks(range(len(cfg.EMOTIONS)))
            ax.set_yticklabels(cfg.EMOTIONS)
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.set_title(title); plt.tight_layout()
        out_path = os.path.join('/kaggle/working', fname)
        plt.savefig(out_path, dpi=150); plt.close()
        print(f"  Confusion matrix → {out_path}")

    def save_training_curves(self, fname='training_curves.png'):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        ep = range(1, len(self.hist['tl']) + 1)
        axes[0].plot(ep, self.hist['tl'], marker='o')
        axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
        axes[0].grid(True)
        axes[1].plot(ep, self.hist['ta'], marker='o', label='Train WA')
        axes[1].plot(ep, self.hist['va'], marker='s', label='Val WA')
        axes[1].set_title('Weighted Accuracy (%)'); axes[1].legend()
        axes[1].grid(True)
        plt.tight_layout()
        out_path = os.path.join('/kaggle/working', fname)
        plt.savefig(out_path, dpi=150); plt.close()
        print(f"  Training curves  → {out_path}")

    # training loop
    def train(self, train_loader: DataLoader,
              val_loader: DataLoader) -> dict:
        for ep in range(1, self.c.EPOCHS + 1):
            tl, ta = self.train_epoch(train_loader, ep)
            va, vf, _, _ = self.evaluate(val_loader, f'Val ep{ep}')
            self.sched.step(va)
            for k, v in zip(['tl','ta','va','vf'], [tl, ta, va, vf]):
                self.hist[k].append(v)
            if va > self.best_val_wa:
                self.best_val_wa = va
                ckpt = '/kaggle/working/best_model.pth'
                torch.save(self.model.state_dict(), ckpt)
                print(f" New best  val WA = {va:.2f}%  → {ckpt}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache(); gc.collect()
        return self.hist

def verify_paths():
    print("\n" + "═"*65)
    print("  DATASET & PACKAGE CHECK")
    print("═"*65)
    rows = [
        ('CMU-MOSI',  cfg.MOSI_PATH,    'Required — trimodal primary'),
        ('CK+',       cfg.CKP_PATH,     'Auxiliary — visual only'),
        ('RAVDESS',   cfg.RAVDESS_PATH, 'Auxiliary — audio only'),
        ('SemEval18', cfg.SEMEVAL_PATH, 'Auxiliary — text only'),
    ]
    for name, path, note in rows:
        ok = os.path.isdir(path)
        print(f"  {'' if ok else ''} {name:12s}  {note}")
        print(f"    {'Found' if ok else 'NOT FOUND'}: {path}")

    print("\n  Package check:")
    for pkg in ['torch','torchvision','facenet_pytorch',
                'librosa','cv2','sklearn','seaborn']:
        try:
            __import__(pkg)
            print(f"   {pkg}")
        except ImportError:
            print(f" {pkg}  ← missing")
    # torchtext intentionally excluded 
    print("   torchtext  — NOT used (GloVe downloaded directly via urllib)")

    gpu = (torch.cuda.get_device_name(0)
           if torch.cuda.is_available() else 'None (CPU only)')
    print(f"\n  GPU: {gpu}")
    if not torch.cuda.is_available():
        print("  Enable GPU: Kaggle notebook → Settings → "
              "Accelerator → GPU T4 x2")
    print("═"*65 + "\n")


def main():
    
    verify_paths()

    # 1. Feature extractors
    print("═"*65)
    print("  STEP 1 / 4  —  Initialise feature extractors")
    print("═"*65)
    vis = VisualExtractor(cfg.DEVICE)
    aud = AudioExtractor(cfg.DEVICE)
    txt = TextExtractor()
    verify_extractors(vis, aud, txt)

    # 2. Datasets 
    print("═"*65)
    print("  STEP 2 / 4  —  Build datasets")
    print("═"*65)
    train_ds = build_dataset('train')
    val_ds   = build_dataset('val')

    if train_ds is None or len(train_ds) == 0:
        print("\n  ERROR: no training samples found.")
        print("  • Confirm dataset paths in Config")
        print("  • For CMU-MOSI: ensure labels.json exists at MOSI_PATH")
        return

    # Class-weighted balanced sampler + loss 
    print('  Computing class distribution …')
    label_counts = torch.zeros(cfg.NUM_EMOTIONS)
    for s in train_ds:
        lbl = s['label'] if isinstance(s['label'], int) else int(s['label'])
        if 0 <= lbl < cfg.NUM_EMOTIONS:
            label_counts[lbl] += 1
    print('  Train class counts:')
    for i, (nm, cnt) in enumerate(zip(cfg.EMOTIONS, label_counts)):
        print(f'    {nm:10s}: {int(cnt):5d}  ({100*cnt/label_counts.sum():.1f}%)')
    # Inverse-frequency weights, normalised to mean=1
    cls_w = label_counts.sum() / (cfg.NUM_EMOTIONS * label_counts.clamp(min=1))
    cls_w = cls_w / cls_w.mean()
    print(f'  Class weights: {[round(float(w),2) for w in cls_w]}')
    # WeightedRandomSampler — oversample minority classes
    sample_w = torch.tensor(
        [float(cls_w[s['label'] if isinstance(s['label'],int) else int(s['label'])])
         for s in train_ds])
    sampler = torch.utils.data.WeightedRandomSampler(
        sample_w, num_samples=len(sample_w), replacement=True)

    train_loader = DataLoader(
        train_ds, batch_size=cfg.BATCH_SIZE, sampler=sampler,
        num_workers=2, collate_fn=collate_fn,
        pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(
        val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False,
        num_workers=2, collate_fn=collate_fn,
        pin_memory=True, persistent_workers=True)

    # 3. Model 
    print("═"*65)
    print("  STEP 3 / 4  —  Build model")
    print("═"*65)
    model = CrossModalEmotionModel(cfg)
    n_params = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total parameters     : {n_params:,}")
    print(f"  Trainable parameters : {trainable:,}")

    trainer = Trainer(model, cfg, vis, aud, txt,
                      cls_w=cls_w)
    # Wire class weights into CrossEntropyLoss
    trainer.crit = nn.CrossEntropyLoss(weight=cls_w.to(cfg.DEVICE))
    print('  Class-weighted CrossEntropyLoss set')
    history = trainer.train(train_loader, val_loader)
    trainer.save_training_curves()

    # 4. Final evaluation
    print("═"*65)
    print("  STEP 4 / 4  —  Final evaluation on test set")
    print("═"*65)
    ckpt = '/kaggle/working/best_model.pth'
    model.load_state_dict(torch.load(ckpt, map_location=cfg.DEVICE))

    test_ds = build_dataset('test')
    if test_ds and len(test_ds) > 0:
        test_loader = DataLoader(
            test_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
            num_workers=2, collate_fn=collate_fn)
        wa, f1, preds, labs = trainer.evaluate(test_loader, 'Test')
        trainer.save_confusion_matrix(
            preds, labs,
            'Test Confusion Matrix  (normalised)',
            'confusion_matrix_test.png')

    print("\n" + "═"*65)
    print(f"  Best Val WA  : {max(history['va']):.2f}%")
    print(f"  Best Val F1  : {max(history['vf']):.4f}")
    print("═"*65)


if __name__ == '__main__':
    torch.manual_seed(42)
    random.seed(42)
    np.random.seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)
    main()

Device       : cuda
Batch / LR   : 64 / 0.0003
E2E ratio    : 1:7
Visual back  : vggface2
GloVe        : 6B 300-d

═════════════════════════════════════════════════════════════════
  DATASET & PACKAGE CHECK
═════════════════════════════════════════════════════════════════
   CMU-MOSI      Required — trimodal primary
    Found: /kaggle/input/datasets/mathurinache/cmu-mosi
   CK+           Auxiliary — visual only
    Found: /kaggle/input/datasets/shuvoalok/ck-dataset
   RAVDESS       Auxiliary — audio only
    Found: /kaggle/input/datasets/uwrfkaggler/ravdess-emotional-song-audio
   SemEval18     Auxiliary — text only
    Found: /kaggle/input/datasets/context/semeval-2018-task-ec

  Package check:
   torch
   torchvision
 facenet_pytorch  ← missing
   librosa
   cv2
   sklearn
   seaborn
   torchtext  — NOT used (GloVe downloaded directly via urllib)

  GPU: Tesla T4
═════════════════════════════════════════════════════════════════

═══════════════════════════════════════════════════════

100%|██████████| 97.8M/97.8M [00:00<00:00, 171MB/s]


Visual : ResNet-50 ImageNet + 300-d head
 Saved → /kaggle/working/feat_cache/Cnn14_mAP=0.431.pth
 PANNs weights: 75/81 layers loaded (93%)
Audio  : PANNs CNN14 (ACTUAL weights) + log-Mel 128 bins
  URL: https://nlp.stanford.edu/data/glove.6B.zip
  Progress: 98%
 Downloaded → /kaggle/working/glove_cache/glove.6B.zip
  Extracting glove.6B.300d.txt from zip …
 Extracted → /kaggle/working/glove_cache/glove.6B.300d.txt
  Parsing GloVe vectors…
  Parsed 300,000 lines …
 Saving index cache …
 Text   : GloVe 6B 300-d  (400,000 words)
  Visual (video) : (30, 300)  
  Visual (image) : (30, 300)  
  Audio          : (30, 300)  
  Text           : (30, 300)  
  GloVe sanity   : cos_sim(happy, sad) = 0.565  
  All checks passed
═════════════════════════════════════════════════════════════════
  STEP 2 / 4  —  Build datasets
═════════════════════════════════════════════════════════════════
CMU-MOSI  [train]   1604 samples  (trimodal: video+audio+text)
CK+       [train]    648 samples  (visual only, 

Epoch 01 [Warmup 1/3]: 100%|██████████| 143/143 [58:11<00:00, 24.41s/it, acc=23.2%, loss=1.4315] 


  [Val ep1]   WA = 11.49%    F1 = 0.0620
              precision    recall  f1-score   support

       happy       0.00      0.00      0.00       519
         sad       0.00      0.00      0.00       303
       anger       0.00      0.00      0.00       267
    surprise       0.31      0.45      0.37        95
     disgust       0.07      0.82      0.13       106
        fear       0.32      0.24      0.27       146

    accuracy                           0.11      1436
   macro avg       0.12      0.25      0.13      1436
weighted avg       0.06      0.11      0.06      1436

 New best  val WA = 11.49%  → /kaggle/working/best_model.pth


...

Epoch 09 [E2E ep9]: 100%|██████████| 143/143 [00:24<00:00,  5.82it/s, acc=46.1%, loss=6.9070]


  [Val ep9]   WA = 33.43%    F1 = 0.3428
              precision    recall  f1-score   support

       happy       0.79      0.34      0.48       519
         sad       0.33      0.15      0.21       303
       anger       0.39      0.23      0.29       267
    surprise       0.23      0.87      0.36        95
     disgust       0.18      0.47      0.26       106
        fear       0.23      0.44      0.30       146

    accuracy                           0.33      1436
   macro avg       0.36      0.42      0.32      1436
weighted avg       0.48      0.33      0.34      1436

 New best  val WA = 33.43%  → /kaggle/working/best_model.pth


Epoch 10 [E2E ep10]: 100%|██████████| 143/143 [00:25<00:00,  5.63it/s, acc=49.0%, loss=6.8317]


  [Val ep10]   WA = 32.87%    F1 = 0.3393
              precision    recall  f1-score   support

       happy       0.80      0.32      0.46       519
         sad       0.34      0.16      0.22       303
       anger       0.38      0.22      0.28       267
    surprise       0.31      0.57      0.40        95
     disgust       0.19      0.66      0.30       106
        fear       0.19      0.51      0.28       146

    accuracy                           0.33      1436
   macro avg       0.37      0.41      0.32      1436
weighted avg       0.49      0.33      0.34      1436

  Training curves  → /kaggle/working/training_curves.png
═════════════════════════════════════════════════════════════════
  STEP 4 / 4  —  Final evaluation on test set
═════════════════════════════════════════════════════════════════
CMU-MOSI  [test ]    344 samples  (trimodal: video+audio+text)
CK+       [test ]    140 samples  (visual only, flat emotion folders)
RAVDESS   [test ]    111 samples  (audio only, 

  [Test]   WA = 29.92%    F1 = 0.2271
              precision    recall  f1-score   support

       happy       0.87      0.17      0.29       115
         sad       0.00      0.00      0.00        98
       anger       0.40      0.02      0.03       110
    surprise       0.26      0.93      0.41        94
     disgust       0.37      0.48      0.42        81
        fear       0.23      0.31      0.26        97

    accuracy                           0.30       595
   macro avg       0.36      0.32      0.24       595
weighted avg       0.37      0.30      0.23       595

  Confusion matrix → /kaggle/working/confusion_matrix_test.png

═════════════════════════════════════════════════════════════════
  Best Val WA  : 33.43%
  Best Val F1  : 0.3428
═════════════════════════════════════════════════════════════════
